# Notebook 6: Fine-Tuning an SSD Car Detector (Google Colab)

> **This notebook must be run in Google Colab, not locally and not on the
> Raspberry Pi.** In Colab: **Runtime -> Change runtime type -> GPU**
> (a T4 is fine, free tier). Training on a laptop CPU or the Pi would be
> extremely slow/impractical for this task; this notebook fine-tunes a
> pretrained SSD in a few minutes on a free T4.

## Goal

`src/vision/object_detection.py` already exists on the Pi, wired into
`ai_drive.py`, but it was written against a model that didn't exist yet -
its preprocessing and output-parsing are documented, best-effort
**placeholders**. This notebook produces the real thing: it takes a
pretrained SSD (already very good at general object detection, including
"car" as one of COCO's 80 classes) and **fine-tunes** it specifically for
this project's car-detection use case, while teaching the core ideas along
the way:

- **bounding boxes** - how a detection's location is represented,
- **IoU (Intersection over Union)** - how "does this box match reality"
  is actually measured,
- **train / validation / test splits** - why a model must never be judged
  on data it trained on,
- **data augmentation** - artificially growing a small dataset without
  collecting more images,
- **overfitting** - what it looks like in the numbers, not just in theory.

## End product

A file named `car_detection.onnx`, exported from the fine-tuned model,
which you'll copy onto the Pi and load into the placeholder pipeline in
**Notebook 7**. This notebook does not touch the Pi at all - everything
here runs in Colab's cloud VM.


## 1. Check what's already installed, and install what Colab doesn't ship with

### Explanation

Colab ships with `torch` and `torchvision` preinstalled (with CUDA support
already wired up), so there's no need to install those from scratch - just
confirm what's there and that a GPU is actually attached. `fiftyone` (for
downloading a small slice of COCO) and `onnx`/`onnxruntime` (for exporting
and verifying the final model) are **not** preinstalled, so those need an
explicit `pip install`.


In [ ]:
import subprocess
subprocess.run(["python", "-c",
    "import torch, torchvision; "
    "print('torch', torch.__version__); "
    "print('torchvision', torchvision.__version__); "
    "print('CUDA available:', torch.cuda.is_available())"
])


In [ ]:
!pip install -q fiftyone onnx onnxruntime


### Expected output

The first cell should print a torch version, a torchvision version, and
`CUDA available: True` - if it prints `False`, double check
**Runtime -> Change runtime type -> GPU** was actually selected (and that
you didn't hit the free-tier GPU quota; Colab will tell you if so).

`fiftyone`'s installer can print pip dependency-resolver warnings about
other preinstalled packages - these are usually safe to ignore. If imports
fail in a later cell, use **Runtime -> Restart runtime** and re-run from
the top.


## 2. Imports and device setup

### Explanation

Standard imports for the rest of the notebook, plus picking `cuda` as the
device when a GPU is available (and falling back to `cpu` otherwise, so
this at least still runs, just slowly, if the runtime type wasn't changed).
A fixed random seed makes the sampling/shuffling below reproducible run to
run.


In [ ]:
import random
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

random.seed(42)
torch.manual_seed(42)


### Expected output

`Using device: cuda` (or `Using device: cpu` if no GPU runtime was
selected - everything below still works, just much more slowly).


## 3. Load the pretrained SSD

### Explanation

`ssdlite320_mobilenet_v3_large` is a lightweight SSD variant, pretrained on
the full COCO dataset (80 classes, "car" already among them) - a good
match for this project since `object_detection.py` already expects an
SSD-family model and the Pi needs something light enough to run at a
usable frame rate on CPU. Loading it with `weights=...COCO_V1` downloads
the pretrained weights; the next several cells **fine-tune** this model
rather than training a new one from scratch.


In [ ]:
from torchvision.models.detection import (
    ssdlite320_mobilenet_v3_large,
    SSDLite320_MobileNet_V3_Large_Weights,
)

weights = SSDLite320_MobileNet_V3_Large_Weights.COCO_V1
model = ssdlite320_mobilenet_v3_large(weights=weights)
model.to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Loaded SSDLite320-MobileNetV3-Large, COCO-pretrained ({num_params:,} parameters).")


### Expected output

`Loaded SSDLite320-MobileNetV3-Large, COCO-pretrained (...,... parameters).`
with a parameter count in the low millions. The weights download the first
time this runs (a few seconds to ~a minute depending on Colab's network).


## 4. Which class index means "car" in THIS model?

### Explanation

A pretrained COCO model has its own internal list mapping class index ->
class name, and that list should be read from the model's own metadata
rather than assumed - `config.py`'s `CAR_CLASS_ID = 3` is explicitly
documented there as *a reasonable guess, not a verified value*. This cell
reads the real answer directly from `weights.meta`.


In [ ]:
try:
    categories = weights.meta["categories"]
    print("Read category list from weights.meta['categories'].")
except KeyError:
    print("weights.meta had no 'categories' key - using the fallback list below. "
          "VERIFY this against your actual torchvision version's tutorial docs.")
    categories = [
        '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
        'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
        'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
        'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A',
        'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
        'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard',
        'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass', 'cup', 'fork',
        'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli',
        'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
        'potted plant', 'bed', 'N/A', 'dining table', 'N/A', 'N/A', 'toilet', 'N/A',
        'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave',
        'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book', 'clock', 'vase',
        'scissors', 'teddy bear', 'hair drier', 'toothbrush',
    ]

car_label_idx = categories.index("car")
print(f"'car' category index in this model's label space: {car_label_idx}")
print(f"Sanity check: categories[{car_label_idx}] = {categories[car_label_idx]!r}")
print("(src/config.py's CAR_CLASS_ID currently defaults to 3 as a guess - compare "
      "that printed index against 3 once this actually runs.)")


### Expected output / if this errors

`'car' category index in this model's label space: 3` (this is the
expected, commonly-cited COCO index for "car" - if it prints something
different, trust the printout over the guess).

> **Honesty note:** Agent 1 wasn't 100% certain `weights.meta['categories']`
> is the right key in every torchvision version - it's the documented key
> as of when this was written, but torchvision's API has moved things
> around before. If the `except KeyError` branch triggers, that's expected
> and handled by the fallback list, not a bug. Either way, double-check the
> printed `car_label_idx` looks right (`config.py`'s `CAR_CLASS_ID`
> currently guesses `3`) before trusting it downstream.


## 5. Download a small car-focused slice of COCO with FiftyOne

### Explanation

The full COCO dataset is roughly 20GB - far more than this fine-tuning
task needs. FiftyOne's zoo loader can pull just the images that contain a
given class, capped at `max_samples`, which keeps this fast and small:
roughly 40-120 MB of images plus a small metadata cache, not tens of GB.
300 training images and 80 validation/test images is intentionally small -
enough to see real fine-tuning happen in a few minutes, not enough to
produce a production-grade detector.


In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

TRAIN_NAME = "car-detect-train"
VALTEST_NAME = "car-detect-valtest"

for name in (TRAIN_NAME, VALTEST_NAME):
    if fo.dataset_exists(name):
        fo.delete_dataset(name)

train_fo = foz.load_zoo_dataset(
    "coco-2017", split="train", label_types=["detections"], classes=["car"],
    max_samples=300, dataset_name=TRAIN_NAME, shuffle=True, seed=42,
)

valtest_fo = foz.load_zoo_dataset(
    "coco-2017", split="validation", label_types=["detections"], classes=["car"],
    max_samples=80, dataset_name=VALTEST_NAME, shuffle=True, seed=42,
)

print(f"train pool: {len(train_fo)} images")
print(f"val/test pool: {len(valtest_fo)} images")


### Expected output

`train pool: 300 images` and `val/test pool: 80 images` (FiftyOne may print
its own download progress bars above these lines - that's normal). This
cell needs network access and can take a minute or two the first time; if
it errors on a fresh runtime, re-running it usually resumes rather than
re-downloading everything.


## 6. Convert FiftyOne's format into a standard PyTorch `Dataset`

### Explanation

`torchvision`'s detection models expect a `Dataset` that yields
`(image, target)` pairs, where `target` is a dict with `"boxes"` (in
absolute pixel `xyxy` coordinates) and `"labels"`. FiftyOne stores boxes as
`[x, y, width, height]` normalized to `[0, 1]` of the image - this class
does that conversion once, up front, and drops any image that ended up
with zero car boxes after filtering.


In [ ]:
from torch.utils.data import Dataset

class FiftyOneCarDataset(Dataset):
    def __init__(self, fo_dataset, transforms=None):
        self.transforms = transforms
        self.samples = []
        for sample in fo_dataset:
            if sample.ground_truth is None:
                continue
            car_dets = [d for d in sample.ground_truth.detections if d.label == "car"]
            if car_dets:
                self.samples.append((sample.filepath, car_dets))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filepath, car_dets = self.samples[idx]
        img = Image.open(filepath).convert("RGB")
        w, h = img.size

        boxes = []
        for det in car_dets:
            x, y, bw, bh = det.bounding_box
            x1, y1 = x * w, y * h
            x2, y2 = (x + bw) * w, (y + bh) * h
            boxes.append([x1, y1, x2, y2])

        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.full((len(boxes),), car_label_idx, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target


### Expected output

No output - this just defines the class. It gets instantiated in Section 7.

**Known, accepted simplification:** these are still full COCO photos -
other real objects (people, other vehicles) may appear in-frame unlabeled
since only `"car"` boxes are kept. That's fine for a short teaching
fine-tune focused specifically on car detection, but it means the model
never gets an explicit "this is NOT a car" signal for those other objects
beyond the implicit background class.


## 7. Train / validation / test split

### Explanation

Three different roles, easy to blur together if you haven't internalized
why they're separate:

- **TRAIN**: what the model's weights are actually updated on.
- **VALIDATION**: checked every epoch during training but never trained
  on - used to watch for overfitting.
- **TEST**: touched exactly once, at the very end, after all
  training/tuning decisions are final - an unbiased estimate of real-world
  performance. If you peek at test results while still tuning, it quietly
  becomes another validation set and stops being trustworthy.

The 80-image validation/test pool downloaded in Section 5 is split 70/30
here into validation and test.


In [ ]:
raw_train_fo = FiftyOneCarDataset(train_fo)

valtest_dataset_raw = FiftyOneCarDataset(valtest_fo)
n_valtest = len(valtest_dataset_raw)
n_val = int(n_valtest * 0.7)
n_test = n_valtest - n_val

generator = torch.Generator().manual_seed(42)
val_subset, test_subset = torch.utils.data.random_split(
    valtest_dataset_raw, [n_val, n_test], generator=generator
)

print(f"train: {len(raw_train_fo)} images")
print(f"val:   {len(val_subset)} images")
print(f"test:  {len(test_subset)} images")


### Expected output

Three counts, e.g. `train: ~290 images`, `val: ~39 images`,
`test: ~17 images` (exact numbers depend on how many of the 300/80
downloaded images actually contained a car box after Section 6's
filtering - slightly fewer than 300/80 is expected and fine).


## 8. See a detection, don't just read about one

### Explanation

Bounding boxes, labels, confidence scores - abstract until you actually
look at one. This draws the ground-truth car box(es) for one training
image next to the **pretrained, not-yet-fine-tuned** model's own
predictions on that same image, side by side.


In [ ]:
def draw_detections(ax, image, boxes, labels_text, scores=None, color="lime"):
    ax.imshow(image)
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        text = labels_text[i] if scores is None else f"{labels_text[i]} {scores[i]:.2f}"
        ax.text(x1, max(y1 - 5, 0), text, color="white", fontsize=9,
                 bbox=dict(facecolor=color, alpha=0.7, pad=1))
    ax.axis("off")

sample_img, sample_target = raw_train_fo[0]
gt_boxes = sample_target["boxes"].tolist()
gt_labels = ["car"] * len(gt_boxes)
print(f"Ground-truth boxes (xyxy, pixel coords): {gt_boxes}")
print(f"Ground-truth labels: {gt_labels}")

model.eval()
img_tensor = torchvision.transforms.functional.to_tensor(sample_img).to(device)
with torch.no_grad():
    prediction = model([img_tensor])[0]

keep = prediction["scores"] > 0.5
pred_boxes = prediction["boxes"][keep].cpu().tolist()
pred_labels = [categories[i] for i in prediction["labels"][keep].cpu().tolist()]
pred_scores = prediction["scores"][keep].cpu().tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
draw_detections(axes[0], sample_img, gt_boxes, gt_labels, color="lime")
axes[0].set_title("Ground truth (no confidence - it's just the label)")
draw_detections(axes[1], sample_img, pred_boxes, pred_labels, pred_scores, color="orange")
axes[1].set_title("Pretrained model's prediction (label + confidence score)")
plt.show()


### Expected output

Two printed lists (ground-truth boxes/labels), then a matplotlib figure
with two panels: green ground-truth box(es) on the left, orange predicted
box(es) with confidence scores on the right. Since this model is already
COCO-pretrained (car was one of its original 80 classes), it should
already find the car reasonably well here, even before any fine-tuning -
fine-tuning below is about specializing it further, not teaching it "car"
from nothing.


## 9. IoU from scratch

### Explanation

**Intersection over Union** is the metric that decides whether a predicted
box "counts" as matching a ground-truth box: the area where the two boxes
overlap, divided by the total area they cover between them. `IoU = 1.0`
means a perfect match; `IoU = 0.0` means no overlap at all. This is
implemented from scratch here (rather than imported) specifically so
there's no hidden magic in the number used to evaluate the model later in
this notebook.


In [ ]:
def compute_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
    inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union_area = area_a + area_b - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


example_pairs = {
    "identical boxes": ([10, 10, 110, 110], [10, 10, 110, 110]),
    "heavy overlap":    ([10, 10, 110, 110], [30, 30, 130, 130]),
    "no overlap":        ([10, 10, 60, 60], [200, 200, 260, 260]),
}

fig, axes = plt.subplots(1, len(example_pairs), figsize=(14, 4))
for ax, (name, (box_a, box_b)) in zip(axes, example_pairs.items()):
    iou = compute_iou(box_a, box_b)
    ax.set_xlim(0, 300); ax.set_ylim(300, 0); ax.set_aspect("equal")
    ax.add_patch(patches.Rectangle((box_a[0], box_a[1]), box_a[2]-box_a[0], box_a[3]-box_a[1],
                                    edgecolor="lime", facecolor="none", linewidth=2))
    ax.add_patch(patches.Rectangle((box_b[0], box_b[1]), box_b[2]-box_b[0], box_b[3]-box_b[1],
                                    edgecolor="orange", facecolor="none", linewidth=2))
    ax.set_title(f"{name}\nIoU = {iou:.3f}")
plt.tight_layout()
plt.show()


### Expected output

Three small panels: `identical boxes` -> `IoU = 1.000`, `heavy overlap` ->
some value between 0 and 1 (around 0.2-0.3 for this exact pair), `no
overlap` -> `IoU = 0.000`. `compute_iou` and these numbers are reused
directly by `evaluate_mean_iou` in Section 13.


## 10. Augmentation with `transforms.v2`

### Explanation

`torchvision.transforms.v2` is the modern augmentation API that knows how
to move bounding boxes correctly when the image itself is flipped or
cropped (the older `transforms` API only transforms images, not the boxes
that go with them). Here: random horizontal flip, color jitter, and a
box-aware random crop (`RandomIoUCrop`), followed by
`SanitizeBoundingBoxes` to drop any box the crop destroyed entirely.

> **This is the most version-sensitive cell in this notebook.** If it
> errors on `SanitizeBoundingBoxes`, `RandomIoUCrop`, or `tv_tensors`,
> it's almost certainly a torchvision version naming difference (older
> versions used `datapoints` instead of `tv_tensors`, or slightly
> different class names) - check Section 1's printed torchvision version,
> search "torchvision transforms v2 [your version] BoundingBoxes", and
> adjust the import/class names accordingly. This isn't a logic bug, it's
> an API-naming issue, and it's the single most likely place in this
> whole notebook to need a small manual fix.


In [ ]:
from torchvision.transforms import v2
from torchvision import tv_tensors

def make_transforms(train):
    base = [v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]
    if not train:
        return v2.Compose(base)
    return v2.Compose([
        v2.RandomHorizontalFlip(p=0.5),
        v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        v2.RandomIoUCrop(),
        v2.SanitizeBoundingBoxes(),
        *base,
    ])

def wrap_target_for_v2(img_pil, target):
    w, h = img_pil.size
    boxes = tv_tensors.BoundingBoxes(target["boxes"], format="XYXY", canvas_size=(h, w))
    return img_pil, {**target, "boxes": boxes}

class TransformWrapper:
    def __init__(self, v2_transform):
        self.v2_transform = v2_transform

    def __call__(self, img_pil, target):
        img_pil, target = wrap_target_for_v2(img_pil, target)
        img_out, target_out = self.v2_transform(img_pil, target)
        target_out["boxes"] = torch.as_tensor(target_out["boxes"], dtype=torch.float32)
        return img_out, target_out

train_transforms = TransformWrapper(make_transforms(train=True))
eval_transforms = TransformWrapper(make_transforms(train=False))


### Expected output / if this errors

No output - this just defines the transform pipeline; Section 11 shows it
running on a real image. If you hit an `AttributeError` or `ImportError`
here, see the version-mismatch note above before assuming the surrounding
logic is wrong - re-run Section 1's version-check cell, search for that
exact torchvision version's transforms.v2 API, and adjust class names as
needed. This is expected to be the most likely single point of friction in
the whole notebook.


## 11. Before/after: what one augmentation pass actually does

### Explanation

Run the training-time transform pipeline once on a real training image and
look at the result directly - the boxes should visibly still line up with
the (possibly flipped/cropped/recolored) car after the transform, proving
the box-tracking actually worked rather than just trusting the API.


In [ ]:
raw_img, raw_target = raw_train_fo[0]
aug_img, aug_target = train_transforms(raw_img, dict(raw_target))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
draw_detections(axes[0], raw_img, raw_target["boxes"].tolist(), ["car"] * len(raw_target["boxes"]))
axes[0].set_title("Original")

aug_img_disp = aug_img.permute(1, 2, 0).numpy()
draw_detections(axes[1], aug_img_disp, aug_target["boxes"].tolist(),
                 ["car"] * len(aug_target["boxes"]))
axes[1].set_title("After flip + color jitter + IoU-crop")
plt.show()


### Expected output

Two panels: the original image with its ground-truth box(es), and an
augmented version (possibly flipped, recolored, and/or cropped in tighter)
with its box(es) still correctly surrounding the car. Since `RandomIoUCrop`
and the flip are random, re-running this cell will show a different
augmentation each time.


## 12. Wrap the splits with their transforms and build `DataLoader`s

### Explanation

Training data gets the augmenting transform pipeline; validation and test
data get only the non-augmenting "convert to tensor" pipeline, since
augmentation is a training-time trick, not something you want when
measuring real performance. `collate_fn` is needed because detection
targets are variable-length (different images have different numbers of
car boxes), so they can't be stacked into a single regular tensor the way
`DataLoader` does by default.


In [ ]:
from torch.utils.data import DataLoader

class SplitWithTransform(Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, target = self.base_dataset[idx]
        return self.transform(img, dict(target))

train_dataset = SplitWithTransform(raw_train_fo, train_transforms)
val_dataset = SplitWithTransform(val_subset, eval_transforms)
test_dataset = SplitWithTransform(test_subset, eval_transforms)

def collate_fn(batch):
    return tuple(zip(*batch))

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2)


### Expected output

No output - this just builds the three loaders used by the training loop
(Section 14) and the evaluation function (Section 13).


## 13. A lightweight validation metric: mean IoU

### Explanation

Full COCO mAP (mean Average Precision) is the standard detection metric,
but it's a fair amount of extra machinery for a short teaching notebook.
Instead: for every ground-truth car box, find the model's best-matching
prediction (by IoU) and average that across all boxes in a split. It's not
a substitute for mAP in a research setting, but it's simple, directly
built on the `compute_iou` from Section 9, and good enough to watch
whether fine-tuning is actually helping.


In [ ]:
def evaluate_mean_iou(model, data_loader, device, score_threshold=0.5, label_idx=car_label_idx):
    model.eval()
    ious = []
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)
            for output, target in zip(outputs, targets):
                gt_boxes = target["boxes"]
                if gt_boxes.numel() == 0:
                    continue
                keep = (output["scores"] >= score_threshold) & (output["labels"] == label_idx)
                pred_boxes = output["boxes"][keep].cpu()
                for gt_box in gt_boxes.tolist():
                    if pred_boxes.numel() == 0:
                        ious.append(0.0)
                        continue
                    best_iou = max(compute_iou(gt_box, pb) for pb in pred_boxes.tolist())
                    ious.append(best_iou)
    return sum(ious) / len(ious) if ious else 0.0


### Expected output

No output - this just defines the function; it's called once per epoch in
the training loop (Section 14) and once more for the final held-out test
evaluation (Section 16).


## 14. Fine-tune: freeze the backbone, train the head

### Explanation

**Transfer learning**: freeze the backbone's pretrained COCO features
(already good general-purpose object features) and fine-tune only the
detection head - fewer trainable parameters, less overfitting risk on our
small ~260-image training set, faster per epoch on the free Colab GPU.
`model(images, targets)` in training mode returns a dict of losses
directly (this is `torchvision`'s detection-model convention); summing
them gives the single scalar to backpropagate.


In [ ]:
for param in model.backbone.parameters():
    param.requires_grad = False

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(trainable_params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)

NUM_EPOCHS = 8

train_losses, val_ious = [], []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        running_loss += losses.item()

    lr_scheduler.step()
    avg_train_loss = running_loss / len(train_loader)
    val_iou = evaluate_mean_iou(model, val_loader, device)

    train_losses.append(avg_train_loss)
    val_ious.append(val_iou)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  train_loss={avg_train_loss:.4f}  val_mean_iou={val_iou:.4f}")


### Expected output

One printed line per epoch, e.g.
`Epoch 1/8  train_loss=1.8421  val_mean_iou=0.5123`, with `train_loss`
generally trending down and `val_mean_iou` generally trending up across
the 8 epochs. On a free T4 this should take roughly a few minutes total,
not hours.

> **What to watch for (overfitting):** train_loss should trend steadily
> down each epoch. If `val_mean_iou` stops improving or gets **worse**
> while `train_loss` keeps dropping, that divergence is the classic
> overfitting signature - the model is memorizing the training images
> rather than learning generalizable car features.


## 15. Plot the curves

### Explanation

A table of per-epoch numbers is harder to read at a glance than a plot -
this puts train loss and validation IoU on the same figure (different
y-axes, since they're different units) so the overfitting signature from
Section 14's note would be immediately visible as two lines that diverge
rather than moving together.


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(range(1, NUM_EPOCHS + 1), train_losses, "o-", color="tab:red", label="train loss")
ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss", color="tab:red")
ax2 = ax1.twinx()
ax2.plot(range(1, NUM_EPOCHS + 1), val_ious, "o-", color="tab:blue", label="val mean IoU")
ax2.set_ylabel("val mean IoU", color="tab:blue")
plt.title("Training loss vs. validation IoU")
plt.show()


### Expected output

A single figure: a red descending-ish curve (train loss) and a blue
ascending-ish curve (val mean IoU). With only 8 epochs and ~260 training
images, expect a noisy but generally improving trend, not a perfectly
smooth textbook curve.


## 16. Final, one-time test-set evaluation

### Explanation

This is the payoff of Section 7's split discipline: `test_loader` has not
been touched anywhere above (not during training, not during any
epoch-by-epoch check) - so this number is the closest thing in this
notebook to an honest estimate of how this model would perform on new,
unseen images.


In [ ]:
test_iou = evaluate_mean_iou(model, test_loader, device)
print(f"Held-out TEST mean IoU (never used during training or epoch-by-epoch tuning): {test_iou:.4f}")


### Expected output

`Held-out TEST mean IoU (never used during training or epoch-by-epoch
tuning): 0.XXXX` - some value, typically in the same rough ballpark as the
last few epochs' `val_mean_iou`, though it can differ somewhat since the
test set is small (only ~15-25 images).


## 17. Export the fine-tuned model to ONNX

### Explanation

This is the critical integration point with `src/vision/object_detection.py`
on the Pi: ONNX is a portable model format that `onnxruntime` (already
verified working on the Pi - see `object_detection.py`'s docstring) can
load without needing PyTorch installed there at all. `dynamic_axes` lets
the exported model accept different input resolutions rather than locking
in exactly one size.

> **Detection-model ONNX export has historically had rough edges.** If
> `torch.onnx.export` errors here, try `opset_version=11` or a higher
> number as a fallback (edit the cell and re-run) before assuming
> something else is broken.


In [ ]:
model.eval()
model.to("cpu")

dummy_input = [torch.rand(3, 320, 320)]

ONNX_PATH = "car_detection.onnx"

torch.onnx.export(
    model,
    (dummy_input,),
    ONNX_PATH,
    input_names=["input"],
    output_names=["boxes", "labels", "scores"],
    dynamic_axes={
        "input": {1: "height", 2: "width"},
        "boxes": {0: "num_detections"},
        "labels": {0: "num_detections"},
        "scores": {0: "num_detections"},
    },
    opset_version=12,
    do_constant_folding=True,
)
print(f"Exported fine-tuned model to {ONNX_PATH}")


### Expected output / if this errors

`Exported fine-tuned model to car_detection.onnx`, possibly preceded by
some `torch.onnx` tracing warnings (usually harmless). If export itself
fails, try changing `opset_version=12` to `11` (or a newer number) and
re-running just this cell before concluding the model/pipeline is broken -
opset compatibility is a known rough edge for detection models
specifically, not a sign the rest of the notebook did something wrong.


## 18. Verify: load the exported file back and run one REAL inference

### Explanation

Don't assume the export worked and matches expectations - load
`car_detection.onnx` back with `onnxruntime` (the same library
`object_detection.py` uses on the Pi) and inspect its real input/output
spec directly, then run one actual inference on a held-out test image.
This printed spec is exactly what will need to be compared against
`object_detection.py`'s `_preprocess()`/`_parse_ssd_output()` placeholder
assumptions in Notebook 7.


In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

print("=" * 70)
print("ONNX MODEL INPUT/OUTPUT SPEC (verified by loading the file, not assumed)")
print("=" * 70)
print("Inputs:")
for inp in session.get_inputs():
    print(f"  name={inp.name!r}  shape={inp.shape}  dtype={inp.type}")
print("Outputs:")
for out in session.get_outputs():
    print(f"  name={out.name!r}  shape={out.shape}  dtype={out.type}")

sample_img_tensor, _ = test_dataset[0]
input_array = sample_img_tensor.numpy()

raw_outputs = session.run(None, {session.get_inputs()[0].name: input_array})
output_names = [o.name for o in session.get_outputs()]
result = dict(zip(output_names, raw_outputs))

print("=" * 70)
print("REAL INFERENCE RESULT on one held-out test image")
print("=" * 70)
print("boxes (xyxy pixel coords of the resized-internally-320 input):")
print(result["boxes"])
print("labels (indices into the category list from Block 4):")
print(result["labels"])
print("scores (confidence, 0-1):")
print(result["scores"])

print("=" * 70)
print(f"'car' category index in this exported model's label space: {car_label_idx}")
print(f"(categories[{car_label_idx}] == {categories[car_label_idx]!r})")
print("=" * 70)


### Expected output

A printed input/output spec (names, shapes, dtypes), followed by real
`boxes`/`labels`/`scores` arrays from one actual test image, followed by
the `car_label_idx` sanity check. **Keep this printout** - it's the ground
truth Notebook 7 (and, eventually, a real update to
`src/vision/object_detection.py`'s placeholder `_preprocess()`/
`_parse_ssd_output()`) will need to match against.


## 19. Get the model onto the Pi

Download `car_detection.onnx` from Colab's file browser (left sidebar) to
your laptop, then:

```
mkdir -p /home/salem/AI_Car_workshop/raspberry_pi_robot/models
mv ~/Downloads/car_detection.onnx /home/salem/AI_Car_workshop/raspberry_pi_robot/models/car_detection.onnx
scp /home/salem/AI_Car_workshop/raspberry_pi_robot/models/car_detection.onnx admin@192.168.0.130:~/raspberry_pi_robot/models/car_detection.onnx
```

Then continue to **Notebook 7** to deploy and test it on the Raspberry Pi.

**Note:** `src/vision/object_detection.py`'s preprocessing/output-parsing
will very likely need updating to match this model's real (not placeholder)
format - Notebook 7 will help diagnose exactly what needs to change, using
the exact spec printed in Section 18 above.


## Recap

- This notebook ran entirely in **Google Colab**, not locally and not on
  the Pi - training-from-scratch-adjacent work needs a GPU this project's
  laptop and Pi don't have.
- A **pretrained** SSD (already COCO-trained, "car" already one of its 80
  classes) was **fine-tuned**, not trained from scratch - the backbone was
  frozen and only the detection head was updated.
- **IoU** (Section 9) is the metric used throughout to decide whether a
  predicted box "counts" as matching a ground-truth box, and
  **mean IoU** (Section 13) was used as this notebook's lightweight
  validation/test metric in place of full COCO mAP.
- The **train/val/test split** (Section 7) was kept strictly separate -
  the test set (Section 16) was touched exactly once, at the very end.
- **Augmentation** (Sections 10-11) used the modern `transforms.v2` API,
  which is also the most version-sensitive cell in this notebook - see its
  note if it errored.
- The final model was exported to **ONNX** and its real input/output spec
  was verified by loading it back and running one real inference
  (Section 18) - that printout is what Notebook 7 and, eventually,
  `src/vision/object_detection.py`'s placeholder code need to match.


## Exercises

**1. Change the confidence threshold used when drawing predictions.**
Section 8 used `prediction["scores"] > 0.5` to decide which boxes to draw.
Try `0.2` (more boxes, more false positives) and `0.8` (fewer boxes, more
confident but possibly missing real cars) and compare.

**2. Try unfreezing the whole backbone, not just the head.**
Comment out the `for param in model.backbone.parameters(): param.requires_grad = False`
loop in Section 14 (or set `requires_grad = True` instead) and re-run
training. Compare the resulting loss/IoU curves against the frozen-backbone
run - with this little data, does unfreezing help or hurt?

**3. Try a different SSD backbone, if curious.**
`torchvision.models.detection.ssd300_vgg16` is a heavier, older SSD variant
- swap it in for `ssdlite320_mobilenet_v3_large` in Section 3 (matching
weights: `SSD300_VGG16_Weights.COCO_V1`) and compare training time and
final IoU. Note this will likely need a different `dummy_input` size in
Section 17's export (300x300 vs. 320x320).

**4. Increase `max_samples` for a larger training set.**
Section 5 downloaded only 300 training images. Try 600 or 1000 and observe
the effect on validation IoU and on how much the train/val curves in
Section 15 diverge (or don't) - more data is one of the most direct ways
to reduce overfitting.
